### This Notebook: Continued Pretraining (CPT) on Agentic AI Book

This notebook is configured for **continued pretraining** of Gemma 4 on the "Agentic AI frameworks, platforms, protocols, and tools on AWS" book.

**Key differences from instruction fine-tuning:**
- Uses raw text completion (next-token prediction), not chat format
- Trains on ALL tokens (no masking of user vs assistant parts)
- Includes `embed_tokens` and `lm_head` in LoRA for domain adaptation
- Uses `UNSLOTH_RETURN_LOGITS=1` to disable CCE (not supported for CPT)
- Higher LoRA rank (128) and rank-stabilized LoRA for better adaptation
- EOS tokens added to help the model learn proper stopping

**Data source:** 92 pages from the AWS Prescriptive Guidance PDF (OCR'd to markdown)

---

### Installation

In [ ]:
%%capture
try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
except: _numpy = "numpy"; _pil = "pillow"

# Set environment variable to disable CCE for continued pretraining
# CCE (Constant Cross Entropy) is not supported for CPT
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
    unsloth "unsloth_zoo>=2026.4.6" transformers==5.5.0 torchcodec timm

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel
import torch

gemma4_models = [
    # Gemma-4 instruct models:
    "unsloth/gemma-4-E2B-it",
    "unsloth/gemma-4-E4B-it",
    "unsloth/gemma-4-31B-it",
    "unsloth/gemma-4-26B-A4B-it",
    # Gemma-4 base models:
    "unsloth/gemma-4-E2B",
    "unsloth/gemma-4-E4B",
    "unsloth/gemma-4-31B",
    "unsloth/gemma-4-26B-A4B",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B",
    dtype = None, # None for auto detection
    max_seq_length = 8192, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
    device_map = {"": torch.cuda.current_device()}, # Use 2x Tesla T4s on Kaggle
)

In [ ]:
from transformers import TextStreamer

# Helper function for chat inference (for testing before CPT)
def do_gemma_4_chat_inference(messages, max_new_tokens = 128):
    # Use apply_chat_template for chat messages
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    
    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        use_cache = True,
        temperature = 1.0, top_p = 0.95, top_k = 64,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

# Helper function for continued pretraining inference (text completion)
def do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7):
    """For CPT inference - direct text completion"""
    import torch
    from transformers import TextIteratorStreamer
    from threading import Thread
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # FIX: Use underlying tokenizer for text completion
    input_ids = tokenizer.tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=True).to(device)
    attention_mask = torch.ones_like(input_ids)
    
    streamer = TextIteratorStreamer(tokenizer.tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=temperature,
        top_p=0.9,
        top_k=50,
        do_sample=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=2,
    )
    
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    generated_text = ""
    for text in streamer:
        print(text, end="", flush=True)
        generated_text += text
    thread.join()
    print()
    return generated_text

# Let's finetune Gemma 4!

You can finetune the vision and text parts for now through selection - the audio part can also be finetuned - we're working to make it selectable as well!

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # Turn off for text CPT
    finetune_language_layers   = True,   # Keep on for continued pretraining
    finetune_attention_modules = True,   # Good for CPT
    finetune_mlp_modules       = True,   # Keep on always
    
    # Additional modules for continued pretraining
    # embed_tokens and lm_head help the model adapt to new vocabulary/domain
    finetune_embedding_modules = True,   # Enable embedding fine-tuning for CPT
    finetune_lm_head          = True,    # Enable lm_head fine-tuning for CPT

    r = 128,           # Higher rank for CPT (64 recommended vs 8 for chat)
    lora_alpha = 32,   # Alpha = r/4 or r/2 is common for CPT
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = True,  # Rank-stabilized LoRA helps for CPT
)

### Data Prep for Continued Pretraining

For continued pretraining, we load raw text and train the model to predict the next token. We don't use chat templates - just raw text chunks.

Now we'll chunk the text into sequences for training. For continued pretraining, we create overlapping chunks to maximize data usage.

In [ ]:
max_seq_length = 8192  # Retained for tokenizer bounds check

print(f'Dataset size: {len(dataset)} examples')

Let's verify the data by looking at a sample chunk:

In [ ]:
print(f"Sample chunk (first 500 chars):\n{'='*50}\n")
print(dataset[0]["text"][:500])
print(f"\n{'='*50}\n")
print(f"Sample chunk (middle of book, first 500 chars):\n{'='*50}\n")
print(dataset[len(dataset)//2]["text"][:500])

For continued pretraining, we use the raw text directly without chat templates. The model will learn to predict the next token.

In [ ]:
# For continued pretraining, we just use the text as-is
# No chat template formatting needed - this is pure next-token prediction

# Let's see the first example
dataset[0]["text"][:200]

The text is already in the correct format for continued pretraining - we just pass the raw text to the model and it learns to predict the next token.

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def add_eos_token(examples):
    """Add EOS token to each text chunk for proper next-token prediction"""
    texts = examples["text"]
    # Add EOS token to help model learn when to stop generating
    return { "text" : [text + EOS_TOKEN for text in texts] }

# Apply EOS token addition
dataset = dataset.map(add_eos_token, batched = True)

print(f"Dataset features: {dataset.features}")
print(f"Number of examples: {len(dataset)}")
print(f"EOS token: {repr(EOS_TOKEN)}")

The dataset is ready. Each example contains a chunk of text that will be used for next-token prediction training.

In [ ]:
# Look at an example
print(dataset[0]["text"][:300] + "...")

<a name="Train"></a>
### Train the model for Continued Pretraining

For continued pretraining, we train on all tokens (no masking of user vs assistant parts). We train the model to predict the next token in the sequence.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,        # Use ratio instead of steps for CPT
        num_train_epochs = 20,       # Multiple epochs over the book
        learning_rate = 2e-5,       # Lower LR for CPT stability
        logging_steps = 10,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        report_to = "none",
        # Key for CPT: lower learning rate for embeddings
        # This is handled via Unsloth's special handling in the trainer
    ),
)

For continued pretraining, we train on ALL tokens (not just responses). We skip the `train_on_responses_only` step since we want the model to learn the full text distribution.

Let's verify the training setup by checking a tokenized example:

In [ ]:
# Tokenize and check the first example
sample_text = dataset[0]["text"]

# FIX: Use underlying tokenizer for CPT
input_ids = tokenizer.tokenizer.encode(sample_text, truncation=True, max_length=max_seq_length)
decoded = tokenizer.tokenizer.decode(input_ids)

print(f"Original length: {len(sample_text)} chars")
print(f"Tokenized length: {len(input_ids)} tokens")
print(f"\nFirst 200 chars of decoded:\n{decoded[:200]}...")

Verify that labels match the input_ids (next-token prediction):

In [ ]:
# Check that we're training on all tokens
input_ids = tokenizer.tokenizer.encode(dataset[0]["text"], truncation=True, max_length=max_seq_length)
print("Input IDs (first 10):", input_ids[:10])
print("Labels would be the same for next-token prediction")
print(f"\nFor continued pretraining, all {len(input_ids)} tokens will be used for training.")

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

# Let's train the model!

To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference for Continued Pretraining

For text completion inference after continued pretraining, we use the model as a text completion model (not chat). The model will continue text based on the patterns learned from the book.

In [ ]:
# Prepare for inference
FastModel.for_inference(model)

# Test prompts related to the book's topic (Agentic AI on AWS)
test_prompts = [
    "Agentic AI frameworks on AWS include",
    "The Model Context Protocol (MCP) is",
    "When implementing multi-agent systems,",
    "Amazon Bedrock Agents provides",
    "What is Amazon Bedrock?",
    "what is multi-agent systems"
]

# Use the helper function for CPT text completion
prompt = test_prompts[5]
print(f"Prompt: {prompt}")
print("="*50)
print("Generated continuation:")
print("="*50)

do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7)

### Try more prompts

You can test the model with different prompts to see how well it learned the book's content:

In [ ]:
# Test with different prompts
for prompt in test_prompts[0:]:
    print(f"\nPrompt: {prompt}")
    print("-" * 50)
    
    do_gemma_4_text_completion(prompt, max_new_tokens=256, temperature=0.7)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
#model.save_pretrained("gemma_4_lora")  # Local saving
#tokenizer.save_pretrained("gemma_4_lora")
# model.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma_4_lora", token = "YOUR_HF_TOKEN") # Online saving

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "gemma_4_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Gemma-4?",}]
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 512, # Increase for longer outputs!
    use_cache = True,
    # Recommended Gemma-4 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-4-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-4-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-4-finetune", tokenizer,
        token = "YOUR_HF_TOKEN"
    )

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now for all models! For now, you can convert easily to `Q8_0, F16 or BF16` precision. `Q4_K_M` for 4bit will come later!

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "HF_ACCOUNT/gemma_4_finetune",
        tokenizer,
        quantization_method = "Q8_0", # Only Q8_0, BF16, F16 supported
        token = "YOUR_HF_TOKEN",
    )